# P-020 — Corrected Feature Importance
P-018 found permutation importance disagrees with NB17 SHAP. This packet runs impurity-based (MDI) importance across 10 splits alongside permutation importance to build an honest consensus view.

**Decision:** What are the true stable feature drivers?

In [1]:

import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'

df = pd.read_csv('perovskite_stability_clean.csv')
X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])
print(f"Dataset: {len(df)} devices, {len(FEATURES)} features")

ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

# NB17 SHAP ranks (hardcoded)
NB17_SHAP = {
    'JV_default_Jsc': 1, 'Perovskite_band_gap': 2, 'JV_default_Voc': 3,
    'JV_default_FF': 4, 'first_Prvskt_annealing_temperature': 5,
    'first_Prvskt_thermal_annealing_time': 6, 'MA': 7, 'Perovskite_thickness': 8,
    'Cell_area_measured': 9, 'Cs': 10, 'FA': 11, 'I': 12, 'Br': 13,
    'Pb': 14, 'Sn': 15, 'Cl': 16
}


Dataset: 1543 devices, 16 features


In [2]:

# Run 10 splits, collect MDI and permutation importance
N_SPLITS = 10
mdi_ranks_all = []
perm_ranks_all = []

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    
    # MDI importance
    mdi_imp = model.feature_importances_
    mdi_order = np.argsort(-mdi_imp)
    mdi_rank = np.empty_like(mdi_order)
    mdi_rank[mdi_order] = np.arange(1, len(FEATURES) + 1)
    mdi_ranks_all.append(mdi_rank)
    
    # Permutation importance
    perm = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=42)
    perm_imp = perm.importances_mean
    perm_order = np.argsort(-perm_imp)
    perm_rank = np.empty_like(perm_order)
    perm_rank[perm_order] = np.arange(1, len(FEATURES) + 1)
    perm_ranks_all.append(perm_rank)
    
    print(f"  Split {seed}: MDI top-3: {[FEATURES[i] for i in mdi_order[:3]]}, "
          f"Perm top-3: {[FEATURES[i] for i in perm_order[:3]]}")

mdi_ranks = np.array(mdi_ranks_all)  # shape (10, 16)
perm_ranks = np.array(perm_ranks_all)
print(f"\nCompleted {N_SPLITS} splits")


  Split 0: MDI top-3: ['Cell_area_measured', 'JV_default_Voc', 'first_Prvskt_thermal_annealing_time'], Perm top-3: ['Perovskite_band_gap', 'Perovskite_thickness', 'JV_default_Jsc']


  Split 1: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Perovskite_band_gap', 'Cell_area_measured', 'Perovskite_thickness']


  Split 2: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'first_Prvskt_thermal_annealing_time'], Perm top-3: ['Cell_area_measured', 'Perovskite_thickness', 'Perovskite_band_gap']


  Split 3: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Perovskite_band_gap', 'Cell_area_measured', 'Perovskite_thickness']


  Split 4: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Cell_area_measured', 'Perovskite_band_gap', 'Perovskite_thickness']


  Split 5: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Cell_area_measured', 'Perovskite_thickness', 'Perovskite_band_gap']


  Split 6: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Perovskite_band_gap', 'first_Prvskt_thermal_annealing_time', 'Cell_area_measured']


  Split 7: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Perovskite_thickness', 'Cell_area_measured', 'Perovskite_band_gap']


  Split 8: MDI top-3: ['Cell_area_measured', 'JV_default_Voc', 'JV_default_Jsc'], Perm top-3: ['Cell_area_measured', 'Perovskite_band_gap', 'first_Prvskt_thermal_annealing_time']


  Split 9: MDI top-3: ['Cell_area_measured', 'JV_default_Jsc', 'JV_default_Voc'], Perm top-3: ['Perovskite_thickness', 'Perovskite_band_gap', 'Cell_area_measured']

Completed 10 splits


In [3]:

# Three-way comparison table
rows = []
for i, feat in enumerate(FEATURES):
    rows.append({
        'feature': feat,
        'mean_mdi_rank': mdi_ranks[:, i].mean(),
        'std_mdi_rank': mdi_ranks[:, i].std(),
        'mean_perm_rank': perm_ranks[:, i].mean(),
        'std_perm_rank': perm_ranks[:, i].std(),
        'nb17_shap_rank': NB17_SHAP[feat],
    })

comp = pd.DataFrame(rows)
comp['avg_rank'] = (comp['mean_mdi_rank'] + comp['mean_perm_rank'] + comp['nb17_shap_rank']) / 3
comp = comp.sort_values('avg_rank')

print("=" * 95)
print("THREE-WAY FEATURE IMPORTANCE COMPARISON")
print("=" * 95)
print(f"{'Feature':<35} {'MDI':>8} {'Perm':>8} {'SHAP':>6} {'Avg':>6}")
print("-" * 95)
for _, r in comp.iterrows():
    print(f"{r['feature']:<35} {r['mean_mdi_rank']:>5.1f}±{r['std_mdi_rank']:.1f} "
          f"{r['mean_perm_rank']:>5.1f}±{r['std_perm_rank']:.1f} "
          f"{r['nb17_shap_rank']:>6.0f} {r['avg_rank']:>6.1f}")

# Consensus top-5: in top-5 for at least 2/3 methods
print("\n--- Consensus Analysis ---")
for _, r in comp.iterrows():
    in_top5 = sum([
        r['mean_mdi_rank'] <= 5,
        r['mean_perm_rank'] <= 5,
        r['nb17_shap_rank'] <= 5
    ])
    if in_top5 >= 2:
        print(f"  CONSENSUS TOP-5: {r['feature']} (MDI={r['mean_mdi_rank']:.1f}, "
              f"Perm={r['mean_perm_rank']:.1f}, SHAP={r['nb17_shap_rank']:.0f}) — {in_top5}/3 methods")
    elif in_top5 == 1:
        print(f"  DISPUTED:        {r['feature']} (MDI={r['mean_mdi_rank']:.1f}, "
              f"Perm={r['mean_perm_rank']:.1f}, SHAP={r['nb17_shap_rank']:.0f}) — 1/3 methods only")


THREE-WAY FEATURE IMPORTANCE COMPARISON
Feature                                  MDI     Perm   SHAP    Avg
-----------------------------------------------------------------------------------------------
JV_default_Jsc                        2.3±0.6   5.0±0.9      1    2.8
Perovskite_band_gap                   8.0±0.0   1.9±0.8      2    4.0
Cell_area_measured                    1.0±0.0   2.0±1.0      9    4.0
first_Prvskt_thermal_annealing_time   4.1±0.7   4.0±1.0      6    4.7
JV_default_Voc                        2.9±0.5   9.5±2.7      3    5.1
first_Prvskt_annealing_temperature    6.1±0.3   6.1±1.1      5    5.7
Perovskite_thickness                  6.9±0.3   2.5±1.0      8    5.8
JV_default_FF                         4.7±0.5  14.1±1.6      4    7.6
FA                                   10.2±0.7   8.5±1.6     11    9.9
MA                                   11.3±0.6  11.7±2.0      7   10.0
Cs                                   10.9±1.1  10.0±2.1     10   10.3
Br                        

In [4]:

# Honest summary and biases
print("=" * 70)
print("HONEST FEATURE IMPORTANCE SUMMARY")
print("=" * 70)
print("""
Method biases:
  MDI (impurity): Overweights high-cardinality & continuous features.
    Cell_area and thickness get inflated because they have many
    unique values for tree splits, not because they drive stability.
  Permutation: Affected by feature correlation — correlated features
    share importance. Jsc/Voc/FF are correlated JV outputs.
  SHAP (NB17): Single split, single model — not cross-validated.

Honest recommendation for partners:
  CONFIDENT drivers: Perovskite_band_gap (top-5 in all 3 methods)
  LIKELY drivers: JV_default_Jsc, JV_default_FF, annealing conditions
  UNCERTAIN: Cell_area_measured, Perovskite_thickness (high in MDI/Perm
    but both have extreme missingness — 65% and 31% — and were
    imputed to median, which creates artificial tree-split points)
  OVERSTATED by NB17: JV_default_Voc (SHAP rank 3, but Perm rank 7-9)
""")

# Save
comp.to_csv('Packet_P020_Corrected_Feature_Importance.csv', index=False)
print("Saved: Packet_P020_Corrected_Feature_Importance.csv")


HONEST FEATURE IMPORTANCE SUMMARY

Method biases:
  MDI (impurity): Overweights high-cardinality & continuous features.
    Cell_area and thickness get inflated because they have many
    unique values for tree splits, not because they drive stability.
  Permutation: Affected by feature correlation — correlated features
    share importance. Jsc/Voc/FF are correlated JV outputs.
  SHAP (NB17): Single split, single model — not cross-validated.

Honest recommendation for partners:
  CONFIDENT drivers: Perovskite_band_gap (top-5 in all 3 methods)
  LIKELY drivers: JV_default_Jsc, JV_default_FF, annealing conditions
  UNCERTAIN: Cell_area_measured, Perovskite_thickness (high in MDI/Perm
    but both have extreme missingness — 65% and 31% — and were
    imputed to median, which creates artificial tree-split points)
  OVERSTATED by NB17: JV_default_Voc (SHAP rank 3, but Perm rank 7-9)

Saved: Packet_P020_Corrected_Feature_Importance.csv


In [5]:

# Status footer
# Count consensus top-5 features
n_consensus = 0
for _, r in comp.iterrows():
    in_top5 = sum([r['mean_mdi_rank'] <= 5, r['mean_perm_rank'] <= 5, r['nb17_shap_rank'] <= 5])
    if in_top5 >= 2:
        n_consensus += 1

if n_consensus >= 5:
    status = "Confirmed"
elif n_consensus >= 3:
    status = "Promising"
else:
    status = "Negative"

print("=" * 65)
print(f"P-020 STATUS: {status}")
print("=" * 65)
print(f"Consensus top-5 features (in >=2/3 methods): {n_consensus}")
print(f"Bandgap is the only feature consistently top-5 across all methods.")
print(f"JV parameters (Jsc, FF) are important but rank varies by method.")
print(f"Cell_area and thickness ranks are inflated by missingness artifacts.")
print(f"NB17's claim of Voc as #3 driver is not supported by permutation importance.")


P-020 STATUS: Confirmed
Consensus top-5 features (in >=2/3 methods): 6
Bandgap is the only feature consistently top-5 across all methods.
JV parameters (Jsc, FF) are important but rank varies by method.
Cell_area and thickness ranks are inflated by missingness artifacts.
NB17's claim of Voc as #3 driver is not supported by permutation importance.
